<a href="https://colab.research.google.com/github/sandrokhizanishvili/AML_GNN_GMA/blob/main/model_pna_enhanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GINe + GFP Enhanced for Anti-Money Laundering Detection

**Course:** Graph Mining and Applications, Sapienza University of Rome

---

## What This Notebook Does

This is the **split-projection GINe+GFP** model. Instead of concatenating all 77
edge features and feeding them through a single projection into message passing,
this architecture:

- **Encodes** using only the 16 baseline transaction features (GINEConv)
- **Injects GFP** separately at decode time via a dedicated `edge_proj_gfp` (61 → 128)
- **Decoder input:** concat(h[src], h[dst], e_base_seed, e_gfp_seed) = 512 dims

This prevents GFP from polluting message passing where GINe's sum aggregator would
blend structural pre-computed signals with its own learned aggregation.

| Property | Value |
|----------|-------|
| Base architecture | GINe (GINEConv + split-projection edge readout) |
| Node feature dims | 6 (with GFP node stats) |
| Edge feature dims | 16 (MP) + 61 (GFP decode-only) = 77 total |


## 1. Installation

This cell installs the correct versions of PyTorch and PyTorch Geometric for Colab.


If the libraries are already installed correctly, you can skip this cell.

In [1]:
print("🚀 SWITCHING TO PYTORCH 2.8 (FAST MODE)...")

# 1. Uninstall the current mismatching versions
# We remove the one you just spent 15 mins compiling, because re-installing
# the CORRECT version via wheels will take only 30 seconds.
# os.system("pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib")
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib

# 2. Install PyTorch 2.8.0 (with CUDA 12.6 support)
# We specify the version explicitly to match the PyG documentation you found.
print("⬇️ Installing PyTorch 2.8.0...")
# os.system("pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126")
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

# 3. Install Graph Libraries for PyTorch 2.8
# This link matches the table in your screenshot: torch-2.8.0 + cu126
print("⬇️ Installing Graph Libraries (Wheels)...")
# os.system("pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.8.0+cu126.html")
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html

print("="*40)
print("✅ SETUP COMPLETE.")
print("⚠️ YOU MUST RESTART THE RUNTIME NOW (Runtime -> Restart Session)")
print("="*40)




import torch

try:
    import torch_sparse
    sparse_status = "✅ Installed"
    sparse_version = torch_sparse.__version__
except ImportError:
    sparse_status = "❌ Not Found"
    sparse_version = "N/A"

print(f"PyTorch Version:      {torch.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()}")
print(f"Torch Sparse Status:  {sparse_status} ({sparse_version})")

if torch.cuda.is_available() and sparse_status == "✅ Installed":
    print("\nSUCCESS! You are ready to run the training loop.")
else:
    print("\n⚠️ Something is still missing. Did you Restart the Runtime?")



!pip install torch_geometric

🚀 SWITCHING TO PYTORCH 2.8 (FAST MODE)...
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
⬇️ Installing PyTorch 2.8.0...
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 209.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 248.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 151.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 88.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 105.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/

## 2. Imports and Reproducibility

We import all required libraries and fix all random seeds so that results are reproducible across runs.

The main libraries used are:
- **PyTorch** for model definition and training
- **PyTorch Geometric (PyG)** for graph data structures and GNN layers
- **scikit-learn** for evaluation metrics
- **tqdm** for progress bars during training

In [2]:
import os, time, math, random, warnings
from google.colab import drive
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, precision_recall_curve, average_precision_score, matthews_corrcoef, auc
)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch_geometric
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import GINEConv
from torch_geometric.utils import degree
from tqdm import tqdm

print(f'PyTorch: {torch.__version__}  |  PyG: {torch_geometric.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)


PyTorch          : 2.8.0+cu126
PyTorch Geometric: 2.7.0
Device           : cuda


## 3. Loading the Graph Data

The graph was built in `Data_preparation_gfp.ipynb` and saved as three PyG `Data` objects
using the **GFP feature set** (77 edge features, 6 node features). We load them here.

### Why Three Separate Graphs?

We use a **cumulative snapshot** design that follows the paper's protocol:

- `train_graph` contains only the training period edges. All of them are labelled and used for training.
- `val_graph` contains training + validation period edges. Only the new validation edges are evaluated; the training edges provide neighbourhood context for message passing.
- `test_graph` contains all edges from all periods. Only the test period edges are evaluated.

This design is important because financial transactions exist in a temporal context. A suspicious transaction in the test set should be evaluated using knowledge of the account's full history, not just the test period. By including all prior edges in each snapshot, we give the GNN access to that historical context during message passing.

### Data Split (Temporal)

The split is done chronologically to prevent data leakage:
- **Train:** first 60% of time period
- **Validation:** next 20%
- **Test:** final 20%

In [3]:
# # Mount Google Drive where the graph files and model checkpoints are stored

# drive.mount('/content/drive')

# os.chdir('/content/drive/MyDrive/GMA_GNN_AML')

In [4]:
DATA_DIR = '/kaggle/input/datasets/sandrokhizanishvili/gnn-aml-gfp'
train_graph = torch.load(os.path.join(DATA_DIR, 'train_graph_gfp.pt'), weights_only=False)
val_graph   = torch.load(os.path.join(DATA_DIR, 'val_graph_gfp.pt'),   weights_only=False)
test_graph  = torch.load(os.path.join(DATA_DIR, 'test_graph_gfp.pt'),  weights_only=False)

def describe_graph(g, name):
    labels = g.y[g.eval_mask]
    n_pos  = (labels == 1).sum().item()
    n_eval = g.eval_mask.sum().item()
    print(f'{name}: nodes={g.num_nodes:,}  edges={g.edge_index.shape[1]:,}  '
          f'eval={n_eval:,}  laund={n_pos:,} ({100*n_pos/n_eval:.4f}%)  '
          f'node_dim={g.x.shape[1]}  edge_dim={g.edge_attr.shape[1]}')
describe_graph(train_graph, 'train_graph')
describe_graph(val_graph,   'val_graph')
describe_graph(test_graph,  'test_graph')
for g, name in [(train_graph,'train'),(val_graph,'val'),(test_graph,'test')]:
    assert (g.y[g.eval_mask] == -1).sum() == 0
print('Label sanity check passed.')


train_graph:
  Nodes          : 712,684
  Edges (total)  : 4,154,429
  Eval edges     : 4,154,429
  Laundering     : 1,813  (0.0436%)
  Node feat dim  : 5
  Edge feat dim  : 77

val_graph:
  Nodes          : 712,684
  Edges (total)  : 5,539,239
  Eval edges     : 1,384,810
  Laundering     : 827  (0.0597%)
  Node feat dim  : 5
  Edge feat dim  : 77

test_graph:
  Nodes          : 712,684
  Edges (total)  : 6,924,049
  Eval edges     : 1,384,810
  Laundering     : 925  (0.0668%)
  Node feat dim  : 5
  Edge feat dim  : 77

Label sanity check passed -- no -1 values in eval masks.


## 4. Hyperparameters

All training and model hyperparameters are defined here in one place for easy tuning.

### Alignment with the Paper

The following settings match the paper (Appendix E, Table 11, Table 12):
- `NUM_LAYERS = 2` -- number of GNN message passing layers
- `NUM_NEIGHBORS = [100, 100]` -- 100 one-hop and 100 two-hop neighbours sampled per seed edge

### Changes vs the Original PNA+GFP

- `HIDDEN_DIM = 128` (was 64). Both edge projections now expand their inputs into 128-dimensional spaces rather than compressing everything into 64. The decoder concatenates four 128-dim vectors, giving a 512-dim input to the MLP classifier.
- `BASE_EDGE_DIM = 16` -- the first 16 columns of `edge_attr` are the transaction-level baseline features (FX-corrected amount, currency mismatch, payment format OHE, timing). These are used in **both** message passing and decoding.
- `GFP_EDGE_DIM = 61` -- the remaining 61 columns are the GFP structural features (scatter-gather, cycle counts, vertex statistics). These are used **only in decoding**, keeping message passing clean of GFP redundancy.

### Class Imbalance Handling

With a 2,290:1 ratio of legitimate to laundering transactions, the model would simply predict everything as legitimate without correction. We use `pos_weight = 8` in the Binary Cross-Entropy loss, which means every laundering transaction contributes 8x more to the loss than a legitimate one. This forces the model to pay attention to the rare class.

The paper used `pos_weight` in the range (6, 8) for the GNN baselines (Table 11). We use 8 as the upper bound of this range.

In [5]:
NODE_DIM       = train_graph.x.shape[1]           # 6
TOTAL_EDGE_DIM = train_graph.edge_attr.shape[1]   # 77
BASE_EDGE_DIM  = 16
GFP_EDGE_DIM   = TOTAL_EDGE_DIM - BASE_EDGE_DIM   # 61

HIDDEN_DIM    = 128
NUM_LAYERS    = 2
DROPOUT       = 0.3
EPOCHS        = 20
LR            = 1e-3
WEIGHT_DECAY  = 1e-5
NUM_NEIGHBORS = [100, 100]
BATCH_SIZE    = 2048

n_neg = (train_graph.y[train_graph.eval_mask] == 0).sum().item()
n_pos = (train_graph.y[train_graph.eval_mask] == 1).sum().item()
print(f'Train imbalance: {n_neg/n_pos:.0f}:1')
POS_WEIGHT = torch.tensor([8.0], device=device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
print(f'NODE_DIM={NODE_DIM}  BASE_EDGE_DIM={BASE_EDGE_DIM}  GFP_EDGE_DIM={GFP_EDGE_DIM}  HIDDEN_DIM={HIDDEN_DIM}')


Train imbalance  : 2290:1  (1,813 pos / 4,152,616 neg)

Config:
  NODE_DIM=5, BASE_EDGE_DIM=16, GFP_EDGE_DIM=61
  HIDDEN_DIM=128, NUM_LAYERS=2, DROPOUT=0.3
  EPOCHS=10, LR=0.001, BATCH_SIZE=2048
  NUM_NEIGHBORS=[100, 100]
  POS_WEIGHT=8.0


## 5. Model Architecture

### The Core Problem: Classifying Edges, Not Nodes

Standard GNNs produce **node embeddings** through message passing. But our task is to classify **edges** (transactions). This requires a two-step pattern:

1. **Encode** -- run message passing over the neighbourhood graph to build informative node embeddings
2. **Decode** -- for each transaction we want to classify, look up the sender and receiver embeddings and combine them with the transaction's own features to produce a classification

This encode-decode separation is also required by how PyG's `LinkNeighborLoader` works. It keeps two sets of edges in each batch:
- `batch.edge_index` -- context edges used only for message passing (no labels)
- `batch.edge_label_index` -- seed edges that we actually want to classify (have labels)

### Why LayerNorm Instead of BatchNorm

BatchNorm normalises across the entire batch. With many context edges per batch but very few laundering edges on average, the batch mean and variance are completely dominated by legitimate transactions. LayerNorm normalises each node independently across its 128 features, so laundering nodes keep their distinctive activation patterns regardless of the class distribution in the batch.

### PNA Architecture Overview — Split Projection

PNA was introduced by Corso et al. (2020) and is one of the GNN models benchmarked in the paper. Its key contribution is combining multiple aggregation functions with degree-aware scalers to produce more expressive node embeddings than any single aggregator.

The key modification in this notebook is how edge features are handled. Instead of projecting all 77 features together into a single 64-dim vector, we use two separate projections and keep GFP features out of message passing entirely:

- `edge_proj_base: Linear(16 → 128)` -- projects baseline transaction features. Used in **both** `encode()` and `decode()`.
- `edge_proj_gfp: Linear(61 → 128)` -- projects GFP structural features. Used **only** in `decode()`.

This matters because PNA's own aggregators (mean, min, max, std) already implicitly compute structural signals — fan, degree, amount distributions — which overlap with what GFP encodes. Feeding GFP into message passing creates conflicting representations of the same information and wastes capacity. By injecting GFP only at decode time, PNA reasons about graph structure undisturbed, and the decoder gets both the learned structural embeddings and the precomputed GFP patterns as complementary signals.

The full forward pass for one transaction A to B looks like this:

```
INPUT
  x [N, 6]                      -- raw node features for all accounts
  edge_attr [E, 77]             -- full edge features for context transactions
  edge_label_attr [n_seeds, 77] -- full features of the seed transactions

ENCODE (message passing — only baseline features, GFP excluded)
  node_proj:       x [N, 6]              --> h0 [N, 128]
  edge_proj_base:  edge_attr[:, :16]     --> e_base [E, 128]   (expanding: 16 → 128)

  Layer 1 (PNAConv using e_base only):
    For each account v, aggregates over neighbours u using:
      aggregators : mean, min, max, std
      scalers     : identity, amplification, attenuation
    Each (aggregator, scaler) pair produces one message vector.
    All combinations are concatenated and passed through an MLP.
    Then: LayerNorm -> ReLU -> Dropout

  Layer 2 (PNAConv): same as layer 1, using h1
    Produces h2 [N, 128]

DECODE (GFP introduced here for the first time)
  e_base_seed = edge_proj_base(edge_label_attr[:, :16])   --> [n_seeds, 128]
  e_gfp_seed  = edge_proj_gfp(edge_label_attr[:, 16:])    --> [n_seeds, 128]
  edge_emb    = concat(h2[A], h2[B], e_base_seed, e_gfp_seed)  --> [n_seeds, 512]
  logit       = MLP(edge_emb)  -- 512 -> 128 -> 1
  P(laundering) = sigmoid(logit)
```

The four components in the decoder each carry distinct information:
- `h2[A]` -- what kind of sender account A is (2-hop neighbourhood context, learned by PNA)
- `h2[B]` -- what kind of receiver account B is (2-hop neighbourhood context, learned by PNA)
- `e_base_seed` -- this specific transaction's features (FX-corrected amount, currency mismatch, payment format, timing)
- `e_gfp_seed` -- precomputed AML graph patterns for this transaction (scatter-gather, cycle counts, vertex statistics)

### PNA Aggregators and Scalers

The 4 aggregators and 3 scalers produce 12 message vectors per node per layer. The degree-based scalers are particularly important for AML: hub accounts involved in Fan-In or Fan-Out laundering patterns have very different in-degrees than normal accounts, and the amplification/attenuation scalers allow the model to weight messages differently based on this structural information.

The degree histogram required by PNAConv is computed on the training graph in-degree distribution. It is computed once and passed to all PNAConv layers at initialisation time.

In [6]:
def build_mlp(in_dim, hidden_dim, out_dim, num_layers=2, dropout=0.3):
    """
    Build a multi-layer perceptron (MLP) with LayerNorm and Dropout.

    This function is used in two places:
    1. As the update function inside each GINEConv layer
    2. As the final edge classifier in the decoder

    The last linear layer has no activation or normalisation, because
    the output is either fed into the next layer (which has its own activation)
    or is a raw logit that will be passed to sigmoid/BCE loss.

    Parameters
    ----------
    in_dim     : input feature dimension
    hidden_dim : width of intermediate layers
    out_dim    : output feature dimension
    num_layers : total number of linear layers
    dropout    : fraction of features randomly zeroed during training

    Returns
    -------
    nn.Sequential : the constructed MLP
    """
    layers = []
    dims   = [in_dim] + [hidden_dim] * (num_layers - 1) + [out_dim]

    for i in range(len(dims) - 1):
        layers.append(nn.Linear(dims[i], dims[i+1]))
        # Add activation, normalisation, and dropout after every layer except the last
        if i < len(dims) - 2:
            layers.append(nn.ReLU())
            layers.append(nn.LayerNorm(dims[i+1]))
            layers.append(nn.Dropout(dropout))

    return nn.Sequential(*layers)

In [7]:
'''
PNAConv (handles graph structure):
┌──────────────────────────────────────────────────────────────┐
│  For each node v:                                            │
│    For each (aggregator, scaler) pair:                       │
│      msg_k = SCALER_k( AGGREGATOR_k( h[u] for u in N(v) ) ) │
│    agg = concat( msg_1, ..., msg_12 )  -- 12 combinations    │
│                                                              │
│    YOUR mlp(agg):  <-- build_mlp lives here                 │
│    ┌────────────────────────────────────────┐                │
│    │ Linear(64->128) -> ReLU -> LN -> Drop  │                │
│    │ -> Linear(128->64)                     │                │
│    └────────────────────────────────────────┘                │
│                                                              │
│    h_new[v] = mlp output                                    │
└──────────────────────────────────────────────────────────────┘
'''

'\nPNAConv (handles graph structure):\n┌──────────────────────────────────────────────────────────────┐\n│  For each node v:                                            │\n│    For each (aggregator, scaler) pair:                       │\n│      msg_k = SCALER_k( AGGREGATOR_k( h[u] for u in N(v) ) ) │\n│    agg = concat( msg_1, ..., msg_12 )  -- 12 combinations    │\n│                                                              │\n│    YOUR mlp(agg):  <-- build_mlp lives here                 │\n│    ┌────────────────────────────────────────┐                │\n│    │ Linear(64->128) -> ReLU -> LN -> Drop  │                │\n│    │ -> Linear(128->64)                     │                │\n│    └────────────────────────────────────────┘                │\n│                                                              │\n│    h_new[v] = mlp output                                    │\n└──────────────────────────────────────────────────────────────┘\n'

In [8]:
'''

h0[v]
  |
PNAConv:
  agg    = concat[ scaler_k( aggregator_k( h0[u] for u in N(v) ) ) for all k ]
  h_new  = mlp( agg )         <- Linear->ReLU->LN->Drop->Linear
  |
LayerNorm(h_new)               <- normalise per node across 64 features
  |
ReLU                           <- clip negatives, add non-linearity
  |
Dropout(0.3)                   <- randomly zero 30% of features
  |
h1[v]  -- ready for next layer or decode

'''

'\n\nh0[v]\n  |\nPNAConv:\n  agg    = concat[ scaler_k( aggregator_k( h0[u] for u in N(v) ) ) for all k ]\n  h_new  = mlp( agg )         <- Linear->ReLU->LN->Drop->Linear\n  |\nLayerNorm(h_new)               <- normalise per node across 64 features\n  |\nReLU                           <- clip negatives, add non-linearity\n  |\nDropout(0.3)                   <- randomly zero 30% of features\n  |\nh1[v]  -- ready for next layer or decode\n\n'

In [9]:
'''

h0 -> [PNAConv + mlp(ReLU#1)] -> LayerNorm -> ReLU#2 -> Dropout -> h1
h1 -> [PNAConv + mlp(ReLU#1)] -> LayerNorm -> ReLU#2 -> Dropout -> h2
                                                                      |
                                                                   decode()
'''

'\n\nh0 -> [PNAConv + mlp(ReLU#1)] -> LayerNorm -> ReLU#2 -> Dropout -> h1\nh1 -> [PNAConv + mlp(ReLU#1)] -> LayerNorm -> ReLU#2 -> Dropout -> h2\n                                                                      |\n                                                                   decode()\n'

In [10]:
class GINe(nn.Module):
    """
    GINe with split-projection edge readout (GFP decode-only injection).
    Message passing uses only the 16 baseline edge features.
    GFP features (61 dims) are injected exclusively at decode time.
    Decoder: concat(h[src], h[dst], e_base_seed, e_gfp_seed) = 512 dims.
    """
    def __init__(self, node_dim, base_edge_dim, gfp_edge_dim, hidden_dim, num_layers, dropout=0.3):
        super().__init__()
        self.dropout       = dropout
        self.base_edge_dim = base_edge_dim

        self.node_proj      = nn.Linear(node_dim,      hidden_dim)
        self.edge_proj_base = nn.Linear(base_edge_dim, hidden_dim)  # 16 → 128 (MP + decode)
        self.edge_proj_gfp  = nn.Linear(gfp_edge_dim,  hidden_dim)  # 61 → 128 (decode only)

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            mlp = build_mlp(hidden_dim, hidden_dim * 2, hidden_dim, dropout=dropout)
            self.convs.append(GINEConv(mlp, edge_dim=hidden_dim))
            self.norms.append(nn.LayerNorm(hidden_dim))

        # 4 × hidden_dim: h[src], h[dst], e_base_seed, e_gfp_seed
        self.edge_classifier = build_mlp(hidden_dim * 4, hidden_dim, 1, dropout=dropout)

    def encode(self, x, edge_index, edge_attr):
        # Only baseline features fed into message passing
        e_base = F.relu(self.edge_proj_base(edge_attr[:, :self.base_edge_dim]))
        h = F.relu(self.node_proj(x))
        for conv, norm in zip(self.convs, self.norms):
            h = conv(h, edge_index, e_base)
            h = F.dropout(F.relu(norm(h)), p=self.dropout, training=self.training)
        return h

    def decode(self, h, edge_label_index, edge_label_attr):
        src, dst = edge_label_index
        e_base = F.relu(self.edge_proj_base(edge_label_attr[:, :self.base_edge_dim]))
        e_gfp  = F.relu(self.edge_proj_gfp(edge_label_attr[:, self.base_edge_dim:]))
        edge_emb = torch.cat([h[src], h[dst], e_base, e_gfp], dim=-1)
        return self.edge_classifier(edge_emb).squeeze(-1)

    def forward(self, x, edge_index, edge_attr, edge_label_index, edge_label_attr):
        h = self.encode(x, edge_index, edge_attr)
        return self.decode(h, edge_label_index, edge_label_attr)


## 6. Data Loader

### Why We Cannot Load the Full Graph at Once

The training graph has over 4 million edges. Loading all of them for a single forward pass would require several gigabytes of GPU memory just for the activations and gradients. Instead, we use PyG's `LinkNeighborLoader` to process the graph in mini-batches.

For each mini-batch, the loader:
1. Picks a batch of seed edges (the transactions we want to classify)
2. For each seed edge's endpoints, samples a local neighbourhood (100 one-hop + 100 two-hop neighbours)
3. Builds a small subgraph from those sampled neighbours
4. Returns the subgraph with two edge sets:
   - `batch.edge_index`: the neighbourhood context edges (used for message passing)
   - `batch.edge_label_index`: the seed edges (used for classification and loss)

### How Seed Edge Features Are Tracked

`make_loader` returns both the loader and `seed_edge_attr` (the raw features for all seed edges). Inside each batch, `batch.input_id` tells us which seed edges from the full pool ended up in this batch, so we can fetch the correct features with `seed_edge_attr[batch.input_id.cpu()]`.

### Class Imbalance in Batches

With 0.0436% laundering rate and batch size 2,048, each batch sees on average fewer than 1 laundering transaction on average. The `pos_weight=8` in the loss function compensates for this by amplifying the gradient from those rare edges.

In [11]:
def make_loader(graph, shuffle=True, verbose=False):
    """
    Build a LinkNeighborLoader for mini-batch edge classification.

    The loader uses the real class distribution (no oversampling). Class
    imbalance is handled instead through pos_weight in the loss function.

    Returns a tuple of (loader, seed_edge_attr) because the loader itself
    does not carry seed edge features -- they must be looked up separately
    using batch.input_id in the training loop.

    Parameters
    ----------
    graph   : PyG Data object (train_graph, val_graph, or test_graph)
    shuffle : True during training for randomness; False during evaluation
    verbose : if True, print the number of positive and negative seed edges

    Returns
    -------
    loader         : LinkNeighborLoader that yields mini-batches
    seed_edge_attr : raw edge features for all seed edges [n_seeds, 16]
    """
    # Extract only the labelled edges from this graph snapshot.
    # eval_mask marks which edges belong to this split's evaluation set.
    seed_mask       = graph.eval_mask
    seed_edge_index = graph.edge_index[:, seed_mask]  # [2, n_seeds]
    seed_labels     = graph.y[seed_mask].float()       # [n_seeds] -- 0.0 or 1.0
    seed_edge_attr  = graph.edge_attr[seed_mask]       # [n_seeds, 16] -- returned separately

    if verbose:
        n_pos = (seed_labels == 1).sum().item()
        n_neg = (seed_labels == 0).sum().item()
        print(f'  Seed edges : {n_pos + n_neg:,} total')
        print(f'  Laundering : {n_pos:,} ({100*n_pos/(n_pos+n_neg):.4f}%)')

    loader = LinkNeighborLoader(
        data             = graph,           # full graph (for neighbourhood sampling)
        num_neighbors    = NUM_NEIGHBORS,   # [100, 100] matches paper
        edge_label_index = seed_edge_index, # which edges to classify
        edge_label       = seed_labels,     # their labels (0 or 1)
        batch_size       = BATCH_SIZE,      # seed edges per mini-batch
        shuffle          = shuffle,
        num_workers      = 0,               # must be 0 in Colab (multiprocessing issues)
        pin_memory       = False,
    )

    return loader, seed_edge_attr

## 7. Training and Evaluation

### Training Loop (`train_epoch`)

Each epoch iterates over all mini-batches produced by the loader. For each batch:
1. The context subgraph is encoded to produce node embeddings
2. The seed edge features are fetched using `batch.input_id` as an index into `seed_edge_attr`
3. The decoder classifies each seed edge using sender embedding + receiver embedding + edge features
4. BCE loss is computed with `pos_weight=8` to upweight laundering edges
5. Gradients are clipped to prevent instability, then weights are updated

### Evaluation Loop (`evaluate`)

Evaluation collects predicted probabilities and true labels across all batches, then computes metrics at a fixed threshold. The threshold default is 0.5, but this is not optimal for imbalanced data -- threshold tuning is done separately in Section 9.

### Checkpointing Strategy

We checkpoint based on **validation PR-AUC**. PR-AUC is threshold-independent and more informative than ROC-AUC for severely imbalanced datasets, because it focuses on the precision-recall tradeoff for the minority class.

### Learning Rate Schedule

Cosine annealing smoothly reduces the learning rate from the initial value down to `eta_min=1e-5` over the training run. This prevents overshooting at the end of training and typically improves final performance.

In [12]:
def train_epoch(model, graph, optimizer):
    """
    Run one full training epoch over all seed edges in the graph.

    Iterates over mini-batches from the loader. For each batch, performs
    a forward pass, computes loss, and updates model weights.

    Parameters
    ----------
    model     : PNA model instance
    graph     : training graph (train_graph)
    optimizer : Adam optimiser

    Returns
    -------
    float : average BCE loss across all batches in this epoch
    """
    model.train()
    loader, seed_edge_attr = make_loader(graph, shuffle=True, verbose=True)
    total_loss = 0.0
    n_batches  = 0
    pbar       = tqdm(loader, desc='  Training', leave=False)

    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()

        # Fetch the raw edge features for the seed edges in this batch.
        # batch.input_id contains the positions of this batch's seed edges
        # in the full seed pool returned by make_loader.
        # We index on CPU then move to GPU to avoid device mismatch errors.
        seed_attr = seed_edge_attr[batch.input_id.cpu()].to(device)

        # Forward pass: encode context graph, decode seed edges
        logits = model(
            batch.x,
            batch.edge_index,        # context edges for message passing
            batch.edge_attr,         # context edge features
            batch.edge_label_index,  # seed edges to classify
            seed_attr,               # seed edge features
        )

        # Compute loss against true labels (0=legitimate, 1=laundering)
        loss = criterion(logits, batch.edge_label)
        loss.backward()

        # Clip gradients to prevent exploding gradients on the sparse laundering signal
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate(model, graph, threshold=0.5):
    """
    Evaluate the model on a graph snapshot and return classification metrics.

    Uses the real class distribution (no oversampling) so that metrics
    reflect true performance on the original data. The threshold parameter
    controls the boundary between predicted laundering and legitimate.

    Note: threshold=0.5 is used here for monitoring during training.
    The optimal threshold is found separately using find_best_threshold()
    after training completes.

    Parameters
    ----------
    model     : trained PNA model
    graph     : graph to evaluate on (val_graph or test_graph)
    threshold : decision boundary for converting probabilities to predictions

    Returns
    -------
    dict with keys: f1, precision, recall, roc_auc, pr_auc
    """
    model.eval()
    loader, seed_edge_attr = make_loader(graph, shuffle=False)
    all_probs  = []
    all_labels = []
    pbar       = tqdm(loader, desc='  Validation', leave=False)

    for batch in pbar:
        batch     = batch.to(device)
        seed_attr = seed_edge_attr[batch.input_id.cpu()].to(device)

        logits = model(
            batch.x,
            batch.edge_index,
            batch.edge_attr,
            batch.edge_label_index,
            seed_attr,
        )
        # Convert logits to probabilities and collect across batches
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(batch.edge_label.long().cpu().numpy())

    # Concatenate all batches into single arrays
    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    # Apply threshold to get binary predictions
    preds = (all_probs >= threshold).astype(int)

    # Compute minority-class metrics (pos_label=1 means we evaluate on laundering class only)
    f1  = f1_score(all_labels, preds, pos_label=1, zero_division=0)
    pre = precision_score(all_labels, preds, pos_label=1, zero_division=0)
    rec = recall_score(all_labels, preds, pos_label=1, zero_division=0)
    try:
        roc_auc = roc_auc_score(all_labels, all_probs)
        precision_curve, recall_curve, _ = precision_recall_curve(all_labels, all_probs)
        pr_auc = auc(recall_curve, precision_curve)
    except ValueError:
        # This can happen if a batch has no positives -- safe fallback
        roc_auc = float('nan')
        pr_auc  = float('nan')

    return {'f1': f1, 'precision': pre, 'recall': rec, 'roc_auc': roc_auc, 'pr_auc': pr_auc}


def run_training(model, model_name, checkpoint_path, epochs=EPOCHS):
    """
    Full training loop with validation monitoring and model checkpointing.

    Trains the model for the specified number of epochs, evaluating on the
    validation set after each epoch. Saves the model state whenever validation
    PR-AUC improves. At the end, loads the best checkpoint and evaluates on the
    test set.

    We checkpoint on PR-AUC rather than F1 because PR-AUC is threshold-independent
    and more reliable when F1 at threshold=0.5 is often 0.0 early in training
    (the model outputs low probabilities before it has learned to discriminate).

    Parameters
    ----------
    model           : PNA model instance
    model_name      : name string used in printed output
    checkpoint_path : file path to save the best model weights
    epochs          : number of training epochs

    Returns
    -------
    model        : model loaded with best checkpoint weights
    history      : list of dicts with per-epoch metrics
    test_metrics : dict with final test set results
    """
    optimizer = Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # Cosine annealing smoothly reduces learning rate to eta_min over all epochs.
    # This avoids overshooting at the end of training.
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    best_val_pr_auc = 0.0
    best_state      = None
    history         = []

    print(f'\n{"="*60}')
    print(f'Training {model_name}')
    print(f'  Parameters : {sum(p.numel() for p in model.parameters()):,}')
    print(f'  pos_weight : {float(POS_WEIGHT)} (within paper range 6-8)')
    print(f'{"="*60}')

    for epoch in range(1, epochs + 1):
        print(f'\n--- Epoch {epoch}/{epochs} ---')
        t0 = time.time()

        # Training step
        train_loss = train_epoch(model, train_graph, optimizer)
        scheduler.step()  # update the learning rate for the next epoch

        # Validation step
        val_metrics = evaluate(model, val_graph)
        elapsed     = time.time() - t0

        history.append({'epoch': epoch, 'loss': train_loss, **val_metrics})

        # Save model if validation PR-AUC improved
        improved = ''
        if val_metrics['pr_auc'] > best_val_pr_auc:
            best_val_pr_auc = val_metrics['pr_auc']
            # Deep copy the state so future epochs do not overwrite it
            best_state  = {k: v.clone() for k, v in model.state_dict().items()}
            os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
            torch.save(model.state_dict(), checkpoint_path)
            improved = '  --> New Best Model!'

        print(
            f'Result: Train Loss: {train_loss:.4f} | '
            f'Val F1: {val_metrics["f1"]:.4f} | '
            f'Val Pre: {val_metrics["precision"]:.4f} | '
            f'Val Rec: {val_metrics["recall"]:.4f} | '
            f'Val ROC-AUC: {val_metrics["roc_auc"]:.4f} | '
            f'Val PR-AUC: {val_metrics["pr_auc"]:.4f} | '
            f'Time: {elapsed:.1f}s'
            f'{improved}'
        )

    # Restore best checkpoint for final evaluation
    if best_state is not None:
        model.load_state_dict(best_state)

    # Final evaluation on the held-out test set
    test_metrics = evaluate(model, test_graph)
    print(f'\n{"="*60}')
    print(f'Final Test Results -- {model_name}')
    print(f'  F1        : {test_metrics["f1"]:.4f}')
    print(f'  Precision : {test_metrics["precision"]:.4f}')
    print(f'  Recall    : {test_metrics["recall"]:.4f}')
    print(f'  ROC-AUC   : {test_metrics["roc_auc"]:.4f}')
    print(f'  PR-AUC    : {test_metrics["pr_auc"]:.4f}')
    print(f'{"="*60}')

    return model, history, test_metrics

## 8. Training the Model

We initialise PNA and train it for 10 epochs. Each epoch takes roughly 3-4 minutes on a T4 GPU.

**What to expect during training:**

- **Val F1 = 0.0 in early epochs** -- this is normal and not a bug. With only ~7 laundering edges per batch and the model outputting very low probabilities initially, nothing crosses the default threshold of 0.5. PR-AUC is the more informative metric during training because it directly measures the quality of the precision-recall trade-off for the minority class, regardless of threshold.
- **PR-AUC should climb consistently each epoch** -- if it stays flat or drops, something is wrong.
- **Loss decreasing does not necessarily mean the model is learning** -- with 99.96% legitimate edges, a model that predicts zero for everything has very low loss but is completely useless. PR-AUC is the honest metric here.

The model is saved to Google Drive whenever validation PR-AUC improves, so training can be resumed if the Colab session expires.

In [13]:
EPOCHS = 20
torch.manual_seed(SEED)
gine_gfp_enh_model = GINe(
    node_dim      = NODE_DIM,
    base_edge_dim = BASE_EDGE_DIM,
    gfp_edge_dim  = GFP_EDGE_DIM,
    hidden_dim    = HIDDEN_DIM,
    num_layers    = NUM_LAYERS,
    dropout       = DROPOUT,
).to(device)

gine_gfp_enh_model, gine_gfp_enh_history, _ = run_training(
    gine_gfp_enh_model,
    'GINe + GFP enhanced',
    epochs          = EPOCHS,
    checkpoint_path = '/kaggle/working/Models/GINe_gfp_enh/gine_gfp_enh_epochs_20.pt',
)



Training PNA (split-projection)
  Parameters : 963,969
  pos_weight : 8.0 (within paper range 6-8)

--- Epoch 1/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0148 | Val F1: 0.0000 | Val Pre: 0.0000 | Val Rec: 0.0000 | Val ROC-AUC: 0.9666 | Val PR-AUC: 0.0593 | Time: 966.2s  --> New Best Model!

--- Epoch 2/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0127 | Val F1: 0.0191 | Val Pre: 0.8000 | Val Rec: 0.0097 | Val ROC-AUC: 0.9675 | Val PR-AUC: 0.0804 | Time: 986.7s  --> New Best Model!

--- Epoch 3/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0123 | Val F1: 0.1422 | Val Pre: 0.6535 | Val Rec: 0.0798 | Val ROC-AUC: 0.9726 | Val PR-AUC: 0.1463 | Time: 1002.3s  --> New Best Model!

--- Epoch 4/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0120 | Val F1: 0.2032 | Val Pre: 0.3051 | Val Rec: 0.1524 | Val ROC-AUC: 0.9727 | Val PR-AUC: 0.1616 | Time: 976.6s  --> New Best Model!

--- Epoch 5/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


...

Result: Train Loss: 0.0097 | Val F1: 0.2399 | Val Pre: 0.4349 | Val Rec: 0.1657 | Val ROC-AUC: 0.9665 | Val PR-AUC: 0.2023 | Time: 955.3s

--- Epoch 18/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0096 | Val F1: 0.2308 | Val Pre: 0.4433 | Val Rec: 0.1560 | Val ROC-AUC: 0.9653 | Val PR-AUC: 0.1919 | Time: 953.3s

--- Epoch 19/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0094 | Val F1: 0.2286 | Val Pre: 0.4024 | Val Rec: 0.1596 | Val ROC-AUC: 0.9646 | Val PR-AUC: 0.1945 | Time: 958.2s

--- Epoch 20/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0093 | Val F1: 0.2325 | Val Pre: 0.3965 | Val Rec: 0.1644 | Val ROC-AUC: 0.9642 | Val PR-AUC: 0.1939 | Time: 971.3s



Final Test Results -- PNA (split-projection)
  F1        : 0.2471
  Precision : 0.4463
  Recall    : 0.1708
  ROC-AUC   : 0.9713
  PR-AUC    : 0.2127


In [14]:
# EPOCHS = 20

# torch.manual_seed(SEED)
# pna_model = PNAModel(
#     node_dim      = NODE_DIM,
#     base_edge_dim = BASE_EDGE_DIM,
#     gfp_edge_dim  = GFP_EDGE_DIM,
#     hidden_dim    = HIDDEN_DIM,
#     num_layers    = NUM_LAYERS,
#     deg           = deg,
#     dropout       = DROPOUT,
# ).to(device)

# pna_model, pna_history, pna_test_metrics = run_training(
#     pna_model,
#     'PNA (split-projection)',
#     epochs          = EPOCHS,
#     checkpoint_path = '/kaggle/working/Models/PNA_gfp/pna_split_proj_weight_8_epoch_20.pt',
# )

In [15]:
# torch.manual_seed(SEED)
# pna_model = PNAModel(
#     node_dim      = NODE_DIM,
#     base_edge_dim = BASE_EDGE_DIM,
#     gfp_edge_dim  = GFP_EDGE_DIM,
#     hidden_dim    = HIDDEN_DIM,
#     num_layers    = NUM_LAYERS,
#     deg           = deg,
#     dropout       = DROPOUT,
# ).to(device)

# pna_model, pna_history, pna_test_metrics = run_training(
#     pna_model,
#     'PNA (enhanced v2)',
#     epochs          = EPOCHS,
#     checkpoint_path = 'Models/PNA_gfp/pna_enhanced_gfp_weight_8.pt',
# )

In [16]:
# torch.manual_seed(SEED)
# pna_model = PNAModel(
#     node_dim      = NODE_DIM,
#     base_edge_dim = BASE_EDGE_DIM,
#     gfp_edge_dim  = GFP_EDGE_DIM,
#     hidden_dim    = HIDDEN_DIM,
#     num_layers    = NUM_LAYERS,
#     deg           = deg,
#     dropout       = DROPOUT,
# ).to(device)

# pna_model, pna_history, pna_test_metrics = run_training(
#     pna_model,
#     'PNA (enhanced v2)',
#     epochs          = EPOCHS,
#     checkpoint_path = 'Models/PNA_enhanced/enhanced_v2.pt',
# )

In [17]:
# torch.manual_seed(SEED)
# pna_model = PNAModel(
#     node_dim      = NODE_DIM,
#     base_edge_dim = BASE_EDGE_DIM,
#     gfp_edge_dim  = GFP_EDGE_DIM,
#     hidden_dim    = HIDDEN_DIM,
#     num_layers    = NUM_LAYERS,
#     deg           = deg,
#     dropout       = DROPOUT,
# ).to(device)

# pna_model, pna_history, pna_test_metrics = run_training(
#     pna_model,
#     'PNA (enhanced v2)',
#     epochs          = EPOCHS,
#     checkpoint_path = 'Models/PNA_enhanced/enhanced_v2.pt',
# )

In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gine_gfp_enh_model = GINe(
    node_dim=NODE_DIM, base_edge_dim=BASE_EDGE_DIM, gfp_edge_dim=GFP_EDGE_DIM,
    hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT,
).to(device)
checkpoint_path = '/kaggle/working/Models/GINe_gfp_enh/gine_gfp_enh_epochs_20.pt'
if os.path.exists(checkpoint_path):
    gine_gfp_enh_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print('Model loaded.')
else:
    print('Checkpoint not found.')


Loading model from /kaggle/working/Models/PNA_gfp_enhanced/pna_split_proj_weight_8_epoch_20.pt...
Model loaded successfully.


## 9. Threshold Optimisation

The default decision threshold of 0.5 is rarely optimal for severely imbalanced datasets. Because only 0.05% of transactions are laundering, the model learns to output very low probabilities even for laundering transactions. Using threshold=0.5 means almost nothing gets flagged.

We find two different optimal thresholds, both derived from the **validation set** and then applied to the test set:

1. **Best F1 threshold** -- the threshold that maximises minority-class F1 on the validation set. This matches the paper's evaluation metric and is found analytically from the precision-recall curve.

2. **Best MCC threshold** -- the threshold that maximises Matthews Correlation Coefficient. MCC is a more robust metric for imbalanced data because it considers all four cells of the confusion matrix (TP, FP, FN, TN), while F1 ignores true negatives. MCC is bounded in [-1, +1] where 0 means the model is no better than random.

### Why We Report Multiple Thresholds

Different operational contexts need different thresholds. A bank with a large investigation team can afford a lower threshold (high recall, more false alarms). A bank with a small team needs a higher threshold (high precision, fewer false alarms). Reporting results at multiple thresholds makes the comparison more informative.

In [19]:
@torch.no_grad()
def get_scores(model, graph):
    """
    Run the model on all eval edges in a graph and return raw probabilities.

    This is a lower-level version of evaluate() that returns the raw
    probability scores rather than computing metrics at a fixed threshold.
    Used by find_best_threshold() to sweep over many threshold values.

    Parameters
    ----------
    model : trained PNA model
    graph : graph to score (val_graph or test_graph)

    Returns
    -------
    y_true  : ground truth labels [n_eval] -- integer array
    y_score : predicted probabilities [n_eval] -- float array
    """
    model.eval()
    loader, seed_edge_attr = make_loader(graph, shuffle=False)
    all_probs  = []
    all_labels = []
    pbar       = tqdm(loader, desc='  Scoring', leave=False)

    for batch in pbar:
        batch     = batch.to(device)
        seed_attr = seed_edge_attr[batch.input_id.cpu()].to(device)

        logits = model(
            batch.x,
            batch.edge_index,
            batch.edge_attr,
            batch.edge_label_index,
            seed_attr,
        )
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(batch.edge_label.long().cpu().numpy())

    print('Scoring complete.')
    return np.concatenate(all_labels), np.concatenate(all_probs)


def compute_metrics(y_true, y_score, threshold):
    """
    Compute all classification metrics at a given decision threshold.

    All metrics are computed for the minority class (laundering = 1).
    AUC metrics are threshold-independent and computed from the full
    probability scores.

    Parameters
    ----------
    y_true    : ground truth labels (0 or 1)
    y_score   : predicted probabilities in [0, 1]
    threshold : decision boundary -- predictions above this are classified as laundering

    Returns
    -------
    dict with keys: f1, precision, recall, mcc, roc_auc, pr_auc
    """
    print(f'Computing metrics at threshold {threshold:.4f}...')
    preds = (y_score >= threshold).astype(int)
    precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_score)
    pr_auc = auc(recall_curve, precision_curve)
    metrics = {
        'f1'       : f1_score(y_true, preds, pos_label=1, zero_division=0),
        'precision': precision_score(y_true, preds, pos_label=1, zero_division=0),
        'recall'   : recall_score(y_true, preds, pos_label=1, zero_division=0),
        'mcc'      : matthews_corrcoef(y_true, preds),
        'roc_auc'  : roc_auc_score(y_true, y_score),
        'pr_auc'   : pr_auc,
    }
    print('Metrics computed.')
    return metrics


def find_best_threshold(model, model_name):
    """
    Find the optimal decision thresholds on the validation set and report test results.

    Two thresholds are found:
    1. Best F1 threshold: found analytically from the precision-recall curve
    2. Best MCC threshold: found by sweeping 200 evenly spaced candidates

    Both are found on the validation set and applied to the test set.
    This prevents threshold overfitting to the test set.

    Parameters
    ----------
    model      : trained PNA model
    model_name : name string for display

    Returns
    -------
    best_f1_thr  : optimal F1 threshold (float)
    best_mcc_thr : optimal MCC threshold (float)
    metrics_f1   : test set metrics at the F1 threshold
    metrics_mcc  : test set metrics at the MCC threshold
    """
    y_true_val, y_score_val = get_scores(model, val_graph)

    # Find best F1 threshold analytically via the precision-recall curve.
    # sklearn's precision_recall_curve returns pre, rec, and thresholds for
    # every unique predicted score, so we can find the exact optimal point.
    print('\nFinding best F1 threshold on val set...')
    pre, rec, thresholds = precision_recall_curve(y_true_val, y_score_val)
    f1_scores   = 2 * (pre * rec) / (pre + rec + 1e-8)  # avoid division by zero
    best_f1_idx = f1_scores.argmax()
    best_f1_thr = thresholds[best_f1_idx]
    print(f'  Best F1 threshold : {best_f1_thr:.4f}  '
          f'(Val F1={f1_scores[best_f1_idx]*100:.2f}%)')

    # Find best MCC threshold by sweeping 200 candidate values.
    # We use sklearn's matthews_corrcoef directly on each candidate to guarantee
    # a result in the valid [-1, +1] range. Manual implementations of MCC
    # can overflow with 1.38M samples and produce values outside this range.
    print('\nFinding best MCC threshold on val set...')
    candidates   = np.linspace(y_score_val.min(), y_score_val.max(), 200)
    best_mcc     = -1.0
    best_mcc_thr = candidates[0]

    for thr in tqdm(candidates, desc='  MCC sweep', leave=False):
        preds = (y_score_val >= thr).astype(int)
        mcc   = matthews_corrcoef(y_true_val, preds)
        if mcc > best_mcc:
            best_mcc     = mcc
            best_mcc_thr = float(thr)

    print(f'  Best MCC threshold : {best_mcc_thr:.4f}  (Val MCC={best_mcc:.4f})')
    assert -1.0 <= best_mcc <= 1.0, f'MCC out of range: {best_mcc}'  # sanity check

    print(f'\n{model_name} -- Optimal Thresholds (from val set):')
    print(f'  Best F1  threshold : {best_f1_thr:.4f}  '
          f'(Val F1={f1_scores[best_f1_idx]*100:.2f}%)')
    print(f'  Best MCC threshold : {best_mcc_thr:.4f}  '
          f'(Val MCC={best_mcc:.4f})')

    # Evaluate on the test set using both optimal thresholds and the default 0.5
    y_true_test, y_score_test = get_scores(model, test_graph)
    metrics_f1  = compute_metrics(y_true_test, y_score_test, best_f1_thr)
    metrics_mcc = compute_metrics(y_true_test, y_score_test, best_mcc_thr)
    metrics_05  = compute_metrics(y_true_test, y_score_test, 0.5)

    # Print a clean comparison table
    print(f'\nTest Results')
    print(f"{'Metric':<12} {'thr=0.5':>10} {'Best F1 thr':>12} {'Best MCC thr':>14}")
    print('-' * 52)
    print(f"{'F1 (%)':<12} {metrics_05['f1']*100:>10.2f} {metrics_f1['f1']*100:>12.2f} {metrics_mcc['f1']*100:>14.2f}")
    print(f"{'Precision':<12} {metrics_05['precision']*100:>10.2f} {metrics_f1['precision']*100:>12.2f} {metrics_mcc['precision']*100:>14.2f}")
    print(f"{'Recall':<12} {metrics_05['recall']*100:>10.2f} {metrics_f1['recall']*100:>12.2f} {metrics_mcc['recall']*100:>14.2f}")
    print(f"{'MCC':<12} {metrics_05['mcc']:>10.4f} {metrics_f1['mcc']:>12.4f} {metrics_mcc['mcc']:>14.4f}")
    print(f"{'ROC-AUC':<12} {metrics_05['roc_auc']:>10.4f} {metrics_f1['roc_auc']:>12.4f} {metrics_mcc['roc_auc']:>14.4f}")
    print(f"{'PR-AUC':<12} {metrics_05['pr_auc']:>10.4f} {metrics_f1['pr_auc']:>12.4f} {metrics_mcc['pr_auc']:>14.4f}")
    print(f"{'Threshold':<12} {'0.500':>10} {best_f1_thr:>12.4f} {best_mcc_thr:>14.4f}")

    return best_f1_thr, best_mcc_thr, metrics_f1, metrics_mcc

In [20]:
# Run threshold optimisation on the validation set and evaluate on the test set
pna_f1_thr, pna_mcc_thr, pna_test_f1, pna_test_mcc = find_best_threshold(
    gine_gfp_enh_model, 'PNA + edge readout'
)

Scoring complete.

Finding best F1 threshold on val set...
  Best F1 threshold : 0.5194  (Val F1=25.34%)

Finding best MCC threshold on val set...


  Best MCC threshold : 0.7623  (Val MCC=0.3186)

PNA + edge readout -- Optimal Thresholds (from val set):
  Best F1  threshold : 0.5194  (Val F1=25.34%)
  Best MCC threshold : 0.7623  (Val MCC=0.3186)


Scoring complete.
Computing metrics at threshold 0.5194...
Metrics computed.
Computing metrics at threshold 0.7623...
Metrics computed.
Computing metrics at threshold 0.5000...
Metrics computed.

Test Results
Metric          thr=0.5  Best F1 thr   Best MCC thr
----------------------------------------------------
F1 (%)            24.71        24.66          21.37
Precision         44.63        48.42          80.28
Recall            17.08        16.54          12.32
MCC              0.2758       0.2827         0.3144
ROC-AUC          0.9713       0.9713         0.9713
PR-AUC           0.2127       0.2127         0.2127
Threshold         0.500       0.5194         0.7623


In [21]:
# # Run threshold optimisation on the validation set and evaluate on the test set
# pna_f1_thr, pna_mcc_thr, pna_test_f1, pna_test_mcc = find_best_threshold(
#     gine_gfp_enh_model, 'PNA + edge readout'
# )

In [22]:
# # Run threshold optimisation on the validation set and evaluate on the test set
# pna_f1_thr, pna_mcc_thr, pna_test_f1, pna_test_mcc = find_best_threshold(
#     gine_gfp_enh_model, 'PNA + edge readout'
# )

## 10. Results and Comparison with the Paper

### How to Read the Results

We report three sets of results for the test set:
1. **thr=0.5** -- matches the paper's evaluation protocol exactly (directly comparable)
2. **Best F1 threshold** -- best possible F1 our model can achieve (optimistic but honest)
3. **Best MCC threshold** -- best balanced performance across all four confusion matrix cells

### Understanding the Metrics

**F1 (minority class):** harmonic mean of precision and recall for the laundering class only. This is what the paper reports.

**Precision:** of all transactions flagged as laundering, what fraction actually are?

**Recall:** of all actual laundering transactions, what fraction did the model catch?

**MCC (Matthews Correlation Coefficient):** considers all four cells of the confusion matrix. More robust than F1 for severely imbalanced data. Range: [-1, +1] where 0 = random.

**ROC-AUC:** threshold-independent ranking quality. A value of 0.96 means the model ranks a randomly chosen laundering transaction above a randomly chosen legitimate transaction 96% of the time.

**PR-AUC (Average Precision):** area under the precision-recall curve. More informative than ROC-AUC for severely imbalanced datasets because it focuses on the minority class performance.

### Why F1 Is Low

The paper reports F1 in the range of 17% for PNA on LI-Small at threshold=0.5 (Table 3). With the GFP feature set (77 edge features including base transaction features and GraphFeaturePreprocessor patterns: scatter-gather histograms, temporal cycles, and vertex statistics) we expect improvement over this baseline. PR-AUC and ROC-AUC give a threshold-independent view of how much the richer features help the model discriminate laundering from legitimate transactions.

### Architecture Alignment with the Paper

| Component | Paper | This implementation | Source |
|-----------|-------|---------------------|--------|
| GNN type | PNA | PNAConv | Paper main text |
| Aggregators | mean, min, max, std | mean, min, max, std | Paper main text |
| Scalers | identity, amplification, attenuation | identity, amplification, attenuation | Paper main text |
| Edge readout | concat(h[src], h[dst], e) | concat(h[src], h[dst], e_seed) | Paper main text |
| Layers | 2 | 2 | Paper Appendix E |
| Hidden dim | 64 | 64 | Paper Appendix E |
| Neighbours | [100, 100] | [100, 100] | Paper main text |
| Normalisation | Not specified | LayerNorm (our choice) | Our implementation |
| Learning rate | 0.005-0.05 (tuned range) | 0.001 | Paper Table 11 |
| pos_weight | 6-8 (tuned range) | 8 (upper bound of paper range) | Paper Table 11 |
| Node features | 5 (paper baseline) | 6 (Bank_ID + EntityType OHE) | GFP feature set |
| Edge features | 16 (paper baseline) | 77 (base + GFP structural patterns) | GFP feature set |

In [23]:
# # Compute test metrics at all three thresholds for the results table
# y_true, y_score = get_scores(gine_gfp_enh_model, test_graph)

# metrics_05  = compute_metrics(y_true, y_score, 0.5)
# metrics_f1  = compute_metrics(y_true, y_score, pna_f1_thr)
# metrics_mcc = compute_metrics(y_true, y_score, pna_mcc_thr)

# # Build the results comparison table
# results = pd.DataFrame([
#     {
#         'Model'        : 'PNA + edge readout (ours, thr=0.5)',
#         'F1 (%)'       : round(metrics_05['f1'] * 100, 2),
#         'Precision (%)': round(metrics_05['precision'] * 100, 2),
#         'Recall (%)'   : round(metrics_05['recall'] * 100, 2),
#         'MCC'          : round(metrics_05['mcc'], 4),
#         'ROC-AUC'      : round(metrics_05['roc_auc'], 4),
#         'PR-AUC'       : round(metrics_05['pr_auc'], 4),
#         'Threshold'    : 0.50,
#         'Neighbours'   : '[100, 100]',
#     },
#     {
#         'Model'        : f'PNA + edge readout (ours, best F1 thr={pna_f1_thr:.3f})',
#         'F1 (%)'       : round(metrics_f1['f1'] * 100, 2),
#         'Precision (%)': round(metrics_f1['precision'] * 100, 2),
#         'Recall (%)'   : round(metrics_f1['recall'] * 100, 2),
#         'MCC'          : round(metrics_f1['mcc'], 4),
#         'ROC-AUC'      : round(metrics_f1['roc_auc'], 4),
#         'PR-AUC'       : round(metrics_f1['pr_auc'], 4),
#         'Threshold'    : round(pna_f1_thr, 3),
#         'Neighbours'   : '[100, 100]',
#     },
#     {
#         'Model'        : f'PNA + edge readout (ours, best MCC thr={pna_mcc_thr:.3f})',
#         'F1 (%)'       : round(metrics_mcc['f1'] * 100, 2),
#         'Precision (%)': round(metrics_mcc['precision'] * 100, 2),
#         'Recall (%)'   : round(metrics_mcc['recall'] * 100, 2),
#         'MCC'          : round(metrics_mcc['mcc'], 4),
#         'ROC-AUC'      : round(metrics_mcc['roc_auc'], 4),
#         'PR-AUC'       : round(metrics_mcc['pr_auc'], 4),
#         'Threshold'    : round(pna_mcc_thr, 3),
#         'Neighbours'   : '[100, 100]',
#     },
#     {
#         'Model'        : 'PNA + edge readout (paper, LI-Small)',
#         'F1 (%)'       : '16.45',
#         'Precision (%)': 'N/A',
#         'Recall (%)'   : 'N/A',
#         'MCC'          : 'N/A',
#         'ROC-AUC'      : 'N/A',
#         'PR-AUC'       : 'N/A',
#         'Threshold'    : 0.50,
#         'Neighbours'   : '[100, 100]',
#     },
# ])

# print('\nPNA Results vs Paper (LI-Small)')
# print(results.to_string(index=False))
# print()
# print('Architecture notes (from Altman et al. 2023, Appendix E and Table 11):')
# print('  - PNA with 4 aggregators and 3 scalers : PNAConv(aggregators, scalers, deg)')
# print('  - Edge readout in decoder              : concat(h[src], h[dst], e_seed)')
# print('  - Neighbourhood sampling               : ours=[100,100]  paper=[100,100]')
# print('  - Paper F1 reported at threshold=0.5 on LI-Small (Figure 9b)')
# print('  - MCC uses all four confusion matrix quadrants -- more robust for imbalance')
# print(f'  - Best F1  threshold {pna_f1_thr:.4f} found via PR curve on val set')
# print(f'  - Best MCC threshold {pna_mcc_thr:.4f} found via threshold sweep on val set')

In [24]:
# # Training curves: shows how the model improved over 10 epochs
# fig, axes = plt.subplots(1, 3, figsize=(16, 4))
# fig.suptitle(
#     'PNA + edge readout -- LI-Small  (paper: PNA with edge features, [100,100] neighbours)',
#     fontsize=11, fontweight='bold'
# )

# df_h = pd.DataFrame(pna_history)

# # Plot 1: Validation PR-AUC over epochs
# axes[0].plot(df_h['epoch'], df_h['pr_auc'],
#              color='steelblue', marker='o', markersize=4, label='PNA (ours, [100,100])')
# axes[0].set_title('Validation PR-AUC')
# axes[0].set_xlabel('Epoch')
# axes[0].set_ylabel('PR-AUC')
# axes[0].set_ylim(0.0, 1.0)
# axes[0].legend(fontsize=8)
# axes[0].grid(True, alpha=0.3)

# # Plot 2: Training loss over epochs
# axes[1].plot(df_h['epoch'], df_h['loss'],
#              color='steelblue', marker='o', markersize=4, label='PNA (ours)')
# axes[1].set_title('Training Loss (BCE, pos_weight=8)')
# axes[1].set_xlabel('Epoch')
# axes[1].set_ylabel('Loss')
# axes[1].legend(fontsize=8)
# axes[1].grid(True, alpha=0.3)

# # Plot 3: Precision-Recall curve on test set
# pre_curve, rec_curve, _ = precision_recall_curve(y_true, y_score)
# ap         = average_precision_score(y_true, y_score)
# laund_rate = y_true.mean()  # random classifier baseline = laundering rate

# axes[2].plot(rec_curve, pre_curve,
#              color='steelblue', linewidth=1.5, label=f'PNA (AP={ap:.3f})')
# axes[2].axhline(laund_rate, color='gray', linestyle='--', linewidth=1,
#                 label=f'Random (AP={laund_rate:.4f})')

# # Mark the optimal F1 threshold point on the PR curve
# preds_f1 = (y_score >= pna_f1_thr).astype(int)
# pre_f1   = precision_score(y_true, preds_f1, pos_label=1, zero_division=0)
# rec_f1   = recall_score(y_true, preds_f1, pos_label=1, zero_division=0)
# f1_val   = f1_score(y_true, preds_f1, pos_label=1, zero_division=0)
# axes[2].scatter([rec_f1], [pre_f1], color='red', zorder=5, s=70,
#                 label=f'Optimal thr={pna_f1_thr:.3f}  F1={f1_val*100:.1f}%')

# # Mark the threshold=0.5 point for paper comparison
# preds_05 = (y_score >= 0.5).astype(int)
# pre_05   = precision_score(y_true, preds_05, pos_label=1, zero_division=0)
# rec_05   = recall_score(y_true, preds_05, pos_label=1, zero_division=0)
# f1_05    = f1_score(y_true, preds_05, pos_label=1, zero_division=0)
# axes[2].scatter([rec_05], [pre_05], color='orange', zorder=5, s=70,
#                 label=f'thr=0.5  F1={f1_05*100:.1f}%')

# axes[2].set_title('Precision-Recall Curve (test set)')
# axes[2].set_xlabel('Recall')
# axes[2].set_ylabel('Precision')
# axes[2].legend(fontsize=7)
# axes[2].grid(True, alpha=0.3)

# plt.tight_layout()
# os.makedirs('plots', exist_ok=True)
# plt.savefig('plots/pna_results.png', dpi=150, bbox_inches='tight')
# plt.show()

## 11. Save Predictions for All Splits

In [25]:
@torch.no_grad()
def score_split(model, graph, split_name):
    model.eval()
    loader, seed_edge_attr = make_loader(graph, shuffle=False)
    all_probs, all_labels, all_input_ids = [], [], []
    for batch in tqdm(loader, desc=f'  Scoring {split_name}', leave=False):
        batch = batch.to(device)
        seed_attr = seed_edge_attr[batch.input_id.cpu()].to(device)
        logits = model(batch.x, batch.edge_index, batch.edge_attr,
                       batch.edge_label_index, seed_attr)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(batch.edge_label.long().cpu().numpy())
        all_input_ids.append(batch.input_id.cpu().numpy())
    scores    = np.concatenate(all_probs)
    labels    = np.concatenate(all_labels)
    input_ids = np.concatenate(all_input_ids)
    eval_ei   = graph.edge_index[:, graph.eval_mask]
    eval_et   = graph.edge_time[graph.eval_mask]
    df = pd.DataFrame({
        'split': split_name,
        'src_idx':   eval_ei[0][input_ids].numpy(),
        'dst_idx':   eval_ei[1][input_ids].numpy(),
        'timestamp': eval_et[input_ids].numpy(),
        'score': scores, 'label': labels,
    })
    print(f'  {split_name}: {len(df):,} edges | {(labels==1).sum():,} laund | '
          f'score [{scores.min():.4f}, {scores.max():.4f}]')
    return df

import datetime
date_str = datetime.date.today().strftime('%d_%m_%y')
os.makedirs('/kaggle/working/Models/GINe_gfp_enh/Predictions', exist_ok=True)
df_train = score_split(gine_gfp_enh_model, train_graph, 'train')
df_val   = score_split(gine_gfp_enh_model, val_graph,   'val')
df_test  = score_split(gine_gfp_enh_model, test_graph,  'test')
df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)
out_path = f'/kaggle/working/Models/GINe_gfp_enh/Predictions/gine_gfp_enh_predictions_all_{date_str}.csv'
df_all.to_csv(out_path, index=False)
print(f'Saved -> {out_path}  ({len(df_all):,} rows)')


Scoring all splits...



  train: 4,154,429 edges  |  1,813 laundering  |  score range [0.0000, 0.9981]  |  mean score laund=0.3660  legit=0.0027


  val: 1,384,810 edges  |  827 laundering  |  score range [0.0000, 0.9980]  |  mean score laund=0.2687  legit=0.0030


  test: 1,384,810 edges  |  925 laundering  |  score range [0.0000, 0.9973]  |  mean score laund=0.2716  legit=0.0029

Saved to /kaggle/working/Models/PNA_gfp_enhanced/Predictions/
  predictions_train_pna_enh_gfp_epochs_20_22_05_26.csv : 4,154,429 rows
  predictions_val_pna_enh_gfp_epochs_20_22_05_26.csv   : 1,384,810 rows
  predictions_test_pna_enh_gfp_epochs_20_22_05_26.csv  : 1,384,810 rows
  predictions_all_pna_enh_gfp_epochs_20_22_05_26.csv   : 6,924,049 rows

Sample rows from test set:
split  src_idx  dst_idx  timestamp    score  label
 test   635035   161142 1662653160 0.000004      0
 test   470380   296063 1662653160 0.000006      0
 test   298128   709528 1662653160 0.000013      0
 test   281432   548388 1662653160 0.000008      0
 test   692348   330214 1662653160 0.000029      0
